# 424. Longest Repeating Character Replacement
**Difficulty:** 🟡 Medium · **Topic:** String · **LeetCode:** https://leetcode.com/problems/longest-repeating-character-replacement/

## 💡 Concepts

**Core concept(s):** A **sliding window** that tracks the count of its **most frequent** character.

**Why it applies here:** A window can be made all-one-letter if the number of characters we'd need to change — `window length − (count of the most common letter)` — is at most `k`. Grow the window while that's true; shrink when it isn't.

**Key intuition:** The letters to replace = window size minus how many of the most common letter it holds. Keep that ≤ k.

---

### 📚 What is a Sliding Window?
A **window** is a range `[left, right]` over the string that you grow on the right and shrink on the left, keeping some running summary (a count, a set) as it moves. You never re-scan from scratch.
- **Complexity:** each character enters and leaves the window at most once → **O(n)** total.
- **In Python:** two indices plus a `dict`/`set`/`Counter` describing what's inside.

### 📚 What is a Hash Map / Hash Set?
A **hash map** (Python `dict`) stores **key → value** pairs; a **hash set** (`set`) stores unique keys. Both use a *hash function* to jump straight to a slot instead of scanning.
- **Operations & complexity:** insert / lookup / delete are **O(1) on average**.
- **In Python:** `dict` for counts/mappings, `set` for "have I seen this?". `collections.Counter` counts items for you.

---

**Prerequisite knowledge:**
- Sliding window.
- A count `dict` and tracking the max count.

## 📝 Problem

You may change at most `k` characters to any letter. Return the length of the longest run of the **same** letter you can make.

**Example**
```
s = "ABAB", k = 2  -> 4   (change both A->B or both B->A)
s = "AABABBA", k = 1 -> 4
```

> Two meaningfully distinct approaches: brute `O(n^2)` and sliding window `O(n)`.

### Approach 1 — Check Every Start (worst)

**Idea:** For each start, extend while the window can be fixed with ≤ k changes (`length − most-frequent ≤ k`); stop when it can't (it only gets worse).

**Time complexity:** `O(n^2)`.

**Space complexity:** `O(1)` (26 letters).

In [ ]:
def char_replace_brute(s: str, k: int) -> int:
    best = 0
    n = len(s)
    for i in range(n):                     # try every starting index
        count = {}                         # letter counts inside the current window
        maxf = 0                           # count of the most common letter in the window
        for j in range(i, n):              # extend the window to the right
            count[s[j]] = count.get(s[j], 0) + 1
            maxf = max(maxf, count[s[j]])  # update the most-frequent letter's count
            # letters we'd have to change = window size - most common letter's count
            if (j - i + 1) - maxf > k:     # needs more than k changes...
                break                      # ...and it only gets worse, so stop this start
            best = max(best, j - i + 1)    # window is fixable within k -> record its length
    return best

### Approach 2 — Sliding Window (optimal)

**Idea:** Grow the window; track the most frequent letter's count. If `length − maxCount > k`, shrink from the left. The best window length seen is the answer.

**Time complexity:** `O(n)`.

**Space complexity:** `O(1)`.

In [ ]:
def char_replace_window(s: str, k: int) -> int:
    count = {}                             # letter counts inside the window
    left = best = maxf = 0                 # window left edge; best length; top letter count
    for right in range(len(s)):            # right edge sweeps across
        count[s[right]] = count.get(s[right], 0) + 1
        maxf = max(maxf, count[s[right]])  # most frequent letter in the window
        # If the window needs more than k replacements, shrink it from the left.
        while (right - left + 1) - maxf > k:
            count[s[left]] -= 1            # remove the leftmost letter
            left += 1                      # move the left edge right
        best = max(best, right - left + 1) # window is now valid -> update best length
    return best

In [ ]:
# Correctness check
tests = [("ABAB",2,4), ("AABABBA",1,4), ("AAAA",0,4), ("ABCDE",1,2)]
for s, k, exp in tests:
    a, b = char_replace_brute(s, k), char_replace_window(s, k)
    print(f"{s!r:>10} k={k} -> brute={a}, window={b} | expected={exp}")
    assert a == b == exp, "mismatch!"
print("\nAll tests passed")

## ⏱️ Empirically Checking the Complexities

Big-O can't be read off a function directly, but it can be **measured**. We time each approach on inputs of growing `n` and read the **doubling ratio** — how much runtime grows when `n` doubles.

| Theoretical | Ratio when `n` → `2n` |
|-------------|-----------------------|
| `O(n)`        | ≈ **2×** |
| `O(n log n)`  | ≈ **2×** (slightly more) |
| `O(n²)`       | ≈ **4×** |
| `O(n³)`       | ≈ **8×** |

Inputs are built to force the **worst case** (no early exit). Sub-millisecond rows are noisy — look at the trend.

In [ ]:
import os, sys
_root = os.getcwd()
for _ in range(5):
    if os.path.exists(os.path.join(_root, "bench_utils.py")):
        break
    _root = os.path.dirname(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)
from bench_utils import benchmark   # shared: prints ratio table + optional log-log plot

def make_worst_case(n):
    s = ("AB" * (n // 2 + 1))[:n]           # alternating -> lots of shrinking
    return (s, 1)

solutions = {
    "brute  O(n^2)": char_replace_brute,
    "window O(n)  ": char_replace_window,
}
sizes = [1000, 2000, 4000, 8000]

benchmark(solutions, make_worst_case, sizes, plot=True)


## 🧩 Patterns Learned

- **Window validity via a cheap summary:** track just the most-frequent count; the "changes needed" formula tells you when to shrink.
- **Signal:** "longest substring after changing at most k", "at most k edits / flips".
- **Related problems:** Longest Substring Without Repeating, Max Consecutive Ones III, Minimum Window Substring.
- **Common pitfalls:** (1) recomputing max frequency from scratch (unnecessary); (2) using `if` instead of the window-size check for shrinking.